## 크로스밸리데이션_분류 

In [1]:
import pandas as pd
df = pd.read_csv('wine_data.csv')

In [2]:
features = ['Alcohol', 'Malic', 'Ash', 'Alcalinity', 'Magesium', 'Phenols',
       'Flavanoids', 'Nonflavanoids', 'Proanthocyanins', 'Color', 'Hue',
       'Dilution', 'Proline']

X = df[features]
y = df['class']

In [3]:
# 트레이닝/테스트 데이터 분할
from sklearn.model_selection import train_test_split

X_tn, X_te, y_tn, y_te = train_test_split(X, y, random_state=0)

In [4]:
# 데이터 standardization
from sklearn.preprocessing import StandardScaler
std_scale = StandardScaler()
std_scale.fit(X_tn)

X_tn_std = std_scale.transform(X_tn)
X_te_std = std_scale.transform(X_te)

In [5]:
# 그리드서치 학습
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

param_grid={'kernel': ('linear', 'rbf'), 'C': [0.5, 1, 10, 100]}
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
svc = svm.SVC(random_state=0)
grid_cv = GridSearchCV(svc, param_grid, cv=kfold, scoring='accuracy')
grid_cv.fit(X_tn_std, y_tn)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=0, shuffle=True),
             estimator=SVC(random_state=0),
             param_grid={'C': [0.5, 1, 10, 100], 'kernel': ('linear', 'rbf')},
             scoring='accuracy')

In [6]:
# 그리드 서치 결과 확인
grid_cv.cv_results_

{'mean_fit_time': array([0.000984  , 0.00099955, 0.00080032, 0.00099292, 0.00101347,
        0.00099072, 0.00060687, 0.00080066]),
 'std_fit_time': array([3.22253029e-05, 1.87425514e-05, 4.01263367e-04, 1.53350415e-05,
        1.87878681e-05, 1.34003968e-05, 4.96528900e-04, 4.00842243e-04]),
 'mean_score_time': array([0.00040751, 0.00039449, 0.00060921, 0.0003993 , 0.00039206,
        0.00040131, 0.00039096, 0.00039964]),
 'std_score_time': array([0.00049924, 0.00048323, 0.00049776, 0.00048905, 0.00048028,
        0.0004915 , 0.00047901, 0.00048945]),
 'param_C': masked_array(data=[0.5, 0.5, 1.0, 1.0, 10.0, 10.0, 100.0, 100.0],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=1e+20),
 'param_kernel': masked_array(data=['linear', 'rbf', 'linear', 'rbf', 'linear', 'rbf',
                    'linear', 'rbf'],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object

In [7]:
# 그리드 서치 결과 학인(데이터프레임)
import numpy as np
import pandas as pd

np.transpose (pd.DataFrame(grid_cv.cv_results_))

,0,1,2,3,4,5,6,7
mean_fit_time,0.000984,0.001,0.0008,0.000993,0.001013,0.000991,0.000607,0.000801
std_fit_time,0.000032,0.000019,0.000401,0.000015,0.000019,0.000013,0.000497,0.000401
mean_score_time,0.000408,0.000394,0.000609,0.000399,0.000392,0.000401,0.000391,0.0004
std_score_time,0.000499,0.000483,0.000498,0.000489,0.00048,0.000492,0.000479,0.000489
param_C,0.5,0.5,1.0,1.0,10.0,10.0,100.0,100.0
param_kernel,linear,rbf,linear,rbf,linear,rbf,linear,rbf
params,"{'C': 0.5, 'kernel': 'linear'}","{'C': 0.5, 'kernel': 'rbf'}","{'C': 1, 'kernel': 'linear'}","{'C': 1, 'kernel': 'rbf'}","{'C': 10, 'kernel': 'linear'}","{'C': 10, 'kernel': 'rbf'}","{'C': 100, 'kernel': 'linear'}","{'C': 100, 'kernel': 'rbf'}"
split0_test_score,0.888889,0.962963,0.888889,0.925926,0.888889,0.925926,0.888889,0.925926
split1_test_score,0.962963,1.0,0.962963,0.962963,0.962963,0.962963,0.962963,0.962963
split2_test_score,0.925926,0.962963,0.925926,0.962963,0.925926,0.962963,0.925926,0.962963


In [8]:
#베스트 스코어
grid_cv.best_score_

np.float64(0.9774928774928775)

In [9]:
# 베스트 하이퍼파라미터
grid_cv.best_params_

{'C': 0.5, 'kernel': 'rbf'}

In [10]:
# 최종 모형
clf = grid_cv.best_estimator_
print(clf)

SVC(C=0.5, random_state=0)


In [11]:
# 크로스 밸리데이션 스코어 확인 (1)

from sklearn.model_selection import cross_validate
metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
cv_scores = cross_validate(clf, X_tn_std, y_tn, cv=kfold, scoring=metrics)
cv_scores

{'fit_time': array([0.01024961, 0.        , 0.        , 0.00200009, 0.00200033]),
 'score_time': array([0.00192976, 0.        , 0.0157938 , 0.00399899, 0.00300002]),
 'test_accuracy': array([0.96296296, 1.        , 0.96296296, 0.96153846, 1.        ]),
 'test_precision_macro': array([0.96296296, 1.        , 0.96969697, 0.96969697, 1.        ]),
 'test_recall_macro': array([0.96666667, 1.        , 0.96296296, 0.95833333, 1.        ]),
 'test_f1_macro': array([0.9628483 , 1.        , 0.96451914, 0.96190476, 1.        ])}

In [12]:
# 크로스 밸리데이션 스코어 확인(2)
from sklearn.model_selection import cross_val_score

cv_score = cross_val_score(clf, X_tn_std, y_tn, cv=kfold, scoring='accuracy')
print(cv_score)
print(cv_score.mean())
print(cv_score.std())

[0.96296296 1.         0.96296296 0.96153846 1.        ]
0.9774928774928775
0.01838434849561446


In [13]:
#예측
pred_svm = clf.predict(X_te_std)
print(pred_svm)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 1 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [14]:
# 정확도
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_te, pred_svm)
print(accuracy)

1.0


In [15]:
# 정밀도 (precision)
from sklearn.metrics import precision_score
precision = precision_score(y_te, pred_svm, average='macro')
print(precision)

1.0


In [16]:
# 리콜
from sklearn.metrics import recall_score
recall = recall_score(y_te, pred_svm, average='macro')
print(recall)

1.0


In [17]:
# f1 스코어
from sklearn.metrics import f1_score
f1 = f1_score(y_te, pred_svm, average='macro')
print(f1)

1.0


In [18]:
# confusion matrix 확인 
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(y_te, pred_svm)
print(conf_matrix)

[[16  0  0]
 [ 0 21  0]
 [ 0  0  8]]


In [19]:
# 분류 레포트 확인
from sklearn.metrics import classification_report
class_report = classification_report(y_te, pred_svm)
print(class_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00         8

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45

